In [1]:
import csv
from pathlib import Path
import pandas as pd
from IPython.display import display


def robust_read_csv(path: str) -> pd.DataFrame:
    with open(path, "r") as f:
        lines = [ln.rstrip("\n") for ln in f if ln.strip()]

    if not lines:
        return pd.DataFrame()

    header = next(csv.reader([lines[0]]))
    ncols = len(header)

    rows = []
    for ln in lines[1:]:
        fields = next(csv.reader([ln]))

        if len(fields) > ncols:
            extra = len(fields) - ncols
            name = ",".join(fields[: extra + 1])
            rest = fields[extra + 1 :]
            fields = [name] + rest

        elif len(fields) < ncols:
            fields = fields + [None] * (ncols - len(fields))

        rows.append(fields)

    df = pd.DataFrame(rows, columns=header)

    # 列名正規化
    df.columns = [
        str(c).strip().lower().replace(" ", "_").replace("-", "_")
        for c in df.columns
    ]

    return df


def summarize_single_csv(path: str) -> pd.DataFrame:
    df = robust_read_csv(path)

    if df.empty:
        return pd.DataFrame()

    # 数値変換
    if "success_rate" in df.columns:
        df["success_rate"] = pd.to_numeric(df["success_rate"], errors="coerce")

    if "return" in df.columns:
        df["return"] = pd.to_numeric(df["return"], errors="coerce")

    # 完全空行の除去
    subset_cols = [c for c in ["success_rate", "return"] if c in df.columns]
    if subset_cols:
        df = df.dropna(subset=subset_cols, how="all")

    if "return" in df.columns:
        df = df.nlargest(10, "return")
    
    success_rate_mean = pd.NA
    success_rate_std = pd.NA
    return_mean = pd.NA
    return_std = pd.NA

    if "success_rate" in df.columns:
        success_rate_mean = df["success_rate"].mean()
        success_rate_std = df["success_rate"].std(ddof=0)

    if "return" in df.columns:
        return_mean = df["return"].mean()
        return_std = df["return"].std(ddof=0)

    out = pd.DataFrame(
        [
            {
                "source_csv": Path(path).name,
                "n_rows": len(df),
                "success_rate_mean": success_rate_mean,
                "success_rate_std": success_rate_std,
                "return_mean": return_mean,
                "return_std": return_std,
            }
        ]
    )

    return out


def aggregate_csvs(csv_files: list[str]) -> pd.DataFrame:
    results = []

    for path in csv_files:
        p = Path(path)

        if not p.exists():
            print(f"Warning: file not found -> {path}")
            continue

        summary = summarize_single_csv(path)

        if not summary.empty:
            results.append(summary)

    if not results:
        return pd.DataFrame()

    return pd.concat(results, ignore_index=True)


# =========================
# 使用例
# =========================

csv_files = [
    "/work/robomimic/csv/result/quantize/LNN_standardization/proposal/6-5-6/unit64.csv",
    "/work/robomimic/csv/result/quantize/LNN_standardization/proposal/6-5-6/unit128.csv",
    "/work/robomimic/csv/result/quantize/LNN_standardization/proposal/6-5-6/unit256.csv",
    "/work/robomimic/csv/result/quantize/LNN_standardization/proposal/6-6-6/unit64.csv",
    "/work/robomimic/csv/result/quantize/LNN_standardization/proposal/6-6-6/unit128.csv",
    "/work/robomimic/csv/result/quantize/LNN_standardization/proposal/6-6-6/unit256.csv",
]

df_summary = aggregate_csvs(csv_files)

display(df_summary)

/tmp/ipykernel_4047248/2842062714.py:3: DeprecationWarning: 
Pyarrow will become a required dependency of pandas in the next major release of pandas (pandas 3.0),
(to allow more performant data types, such as the Arrow string type, and better interoperability with other libraries)
but was not found to be installed on your system.
If this would cause problems for you,
please provide us feedback at https://github.com/pandas-dev/pandas/issues/54466
        
  import pandas as pd


,source_csv,n_rows,success_rate_mean,success_rate_std,return_mean,return_std
0,unit64.csv,10,0.939,0.025865,0.939,0.025865
1,unit128.csv,10,0.987,0.009000,0.987,0.009000
2,unit256.csv,10,0.969,0.017000,0.969,0.017000
3,unit64.csv,10,0.962,0.017205,0.962,0.017205
4,unit128.csv,10,0.982,0.010770,0.982,0.010770
5,unit256.csv,10,0.982,0.012490,0.982,0.012490
